<div dir="rtl">
<h1>حلقهٔ تولید را خودتان ببندید</h1>
<p>درس 61 از 76 · چطور از یک پیش‌بینی، متن بلند می‌سازیم؟ · <code dir="ltr">54-generate</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-01/54-generate.html">📖 بازگشت به همین درس</a></p>
<p>از Logits آخرین موقعیت یک ادامهٔ حریصانه بسازید و با generate واقعی مقایسه کنید.</p><p>پیش‌نیاز: شکل (B,T,V)، argmax و بریدن Context.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>وقتی Prompt از ظرفیت Context بلندتر است، طول خروجی باید از Prompt کوتاه‌تر شود یا فقط ورودی forward محدود می‌شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0)).eval()
prompt = torch.tensor([[1,2,3]])
print('actual generation:',model.generate(prompt,3,greedy=True).tolist())
print('This untrained model demonstrates mechanics, not language quality.')

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>greedy_extend(Model, ids, count) را بنویسید. مدلِ ورودی در eval است؛ در no_grad، هر بار آخرین context_length شناسه را بدهید، از آخرین موقعیت argmax بگیرید و به متن کامل اضافه کنید. ids اصلی را تغییر ندهید.</p>
</div>

In [ ]:
def greedy_extend(model, ids, count):
    # TODO: هر بار فقط یک Token تازه
    return None

In [ ]:
def test_exercise():
    result = greedy_extend(model,prompt,3)
    if result is None:
        return False
    assert torch.equal(result,model.generate(prompt,3,greedy=True))
    assert torch.equal(prompt,torch.tensor([[1,2,3]]))
    for ids,count in ((prompt,0),(torch.tensor([[1,2,3,4,5,6]]),2),
                      (torch.tensor([[1,2],[3,4]]),4)):
        assert torch.equal(greedy_extend(model,ids,count),model.generate(ids,count,greedy=True))
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: greedy_extend')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط تعداد Tokenهای تازه را تغییر دهید و تفاوت طول Context و متن برگشتی را ثبت کنید.</p>
</div>

In [ ]:
for count in (0,2,6):
    result = model.generate(prompt,count,greedy=True)
    print(count,'full shape:',tuple(result.shape),'context limit:',model.config.context_length)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>argmax روی محور زمان، شناسهٔ Token نمی‌دهد. last_token_scores(Logits) باید برای ورودی (B,T,V)، امتیازهای آخرین موقعیت با شکل (B,V) را برگرداند.</p>
</div>

In [ ]:
with torch.no_grad():
    scores = model(prompt)[0]
wrong = scores.argmax(dim=1)
print('wrong shape:',tuple(wrong.shape),'these values index time, not vocabulary')

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def last_token_scores(logits):
    # TODO: آخرین موقعیت، همهٔ واژگان
    return None

In [ ]:
def test_repair():
    values = torch.arange(30).reshape(2,3,5)
    result = last_token_scores(values)
    if result is None:
        return False
    assert result.shape==(2,5)
    assert torch.equal(result,torch.tensor([[10,11,12,13,14],[25,26,27,28,29]]))
    assert last_token_scores(torch.zeros(1,1,7)).shape==(1,7)
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: last_token_scores')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>تابع شما با MiniGPT.generate واقعی مقایسه می‌شود. نسخهٔ پروژه علاوه بر حلقه، حالت train/eval قبلی را برمی‌گرداند و گزینه‌های Sampling را اعتبارسنجی می‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا محاسبهٔ امتیاز همهٔ موقعیت‌ها در forward به معنی اضافه‌کردن همهٔ آن پیش‌بینی‌ها به متن نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-01/54-generate.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/54-generate.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>